# core

> Ergonomic wrapper for pandas_gbq that simplifies loading BigQuery data into DataFrames

In [ ]:
#| default_exp core

In [ ]:
#| export
from pandas_gbq import read_gbq as _original_read_gbq, to_gbq as _original_to_gbq, Context, context
import pandas as pd, re, time
from decimal import Decimal
from google.cloud import bigquery


In [ ]:
#| hide
from nbdev.showdoc import *
from google.oauth2 import service_account
import os
import json


In [ ]:
#| export
def read(
    query_or_table:str, # SQL query string or table reference
    verbose:bool=True, # Print timing and DataFrame info
    convert_dtypes:bool=True, # Apply type conversion to columns
    date_cols:list=None, # List of columns to convert to datetime
    str_cols:list=["scv_id"], # List of columns to keep as string/object type
    use_bqstorage_api:bool=True, # Use BigQuery Storage API for faster reads
    **kwargs
    ):
    start = time.time()
    df = _original_read_gbq(query_or_table, use_bqstorage_api=use_bqstorage_api, **kwargs)
    if convert_dtypes: df = convert_bq_dtypes(df, date_cols=date_cols, str_cols=str_cols)
    read_type = 'table' if re.match(r"""^[`\w\-]+\.[\w\-]+.[\w\-\`]+$""", query_or_table) else 'query'
    if verbose:
        elapsed = time.time() - start
        size_gb = get_size_gb(df)
        print(f"Loaded {len(df)} rows × {len(df.columns)} cols ({size_gb:.4f} GB) from {read_type} in {elapsed:.2f}s")
        print(df.info())
    return df

In [ ]:
#| export
def convert_bq_dtypes(
    df:pd.DataFrame, # DataFrame to convert
    date_cols:list=None, # List of columns to convert to datetime
    str_cols:list=None # List of columns to keep as string/object type
):
    "Convert BigQuery data types to pandas-compatible types"
    df = df.copy()
    date_cols = set(date_cols or [])
    str_cols = set(str_cols or [])
    for col in df.columns:
        if col in str_cols: df[col] = df[col].astype('object')
        elif re.search(r"(date|timestamp)", col.lower()) or col in date_cols: df[col] = pd.to_datetime(df[col], errors='coerce')
        elif df[col].dtype == 'float64': df[col] = df[col].astype('Float64')
        elif df[col].dtype == 'int64': df[col] = df[col].astype('Int64')
        elif df[col].dtype == 'object':
            first_val = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
            if first_val is not None and isinstance(first_val, Decimal): df[col] = df[col].astype('Float64')
    return df

In [ ]:
#| export
def to(
    df:pd.DataFrame, # DataFrame to write to BigQuery
    destination_table:str, # Destination table in format 'project.dataset.table' or 'dataset.table'
    verbose:bool=True, # Print timing and data size info
    **kwargs
):
    "Write DataFrame to BigQuery table"
    start = time.time()
    result = _original_to_gbq(df, destination_table, **kwargs)
    if verbose:
        elapsed = time.time() - start
        size_gb = get_size_gb(df)
        print(f"Sent {len(df)} rows × {len(df.columns)} cols ({size_gb:.4f} GB) to {destination_table} in {elapsed:.2f}s")
    return result

In [ ]:
#| export

def get_size_gb(df):
    return df.memory_usage(deep=True).sum() / 1024**3


In [ ]:
#| export
def ex(
    query:str, # SQL query string
    project_id:str='petsathome-sb-datascience', # Project ID
    verbose:bool=True, # Print timing and processing info
    **kwargs
    ):
    "Execute query in BigQuery without returning results"
    client = bigquery.Client(project=project_id, **kwargs)
    start = time.time()
    job = client.query(query)
    result = job.result()
    if verbose:
        elapsed = time.time() - start
        gb_processed = (job.total_bytes_processed or 0) / 1024**3
        rows_affected = job.num_dml_affected_rows if job.num_dml_affected_rows else 0
        cached = " (cached)" if job.cache_hit else ""
        print(f"Processed {gb_processed:.4f} GB{cached}, {rows_affected} rows affected in {elapsed:.2f}s")
    return result


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
